# Sequence models meet the real world: time-series forecasting with LSTM

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudanum/MIA5150/blob/main/notebooks/sequential-models-rnn-gru-lstm/06_time_series_forecasting.ipynb)

Up to now our sequence models have worked on synthetic or text data. In this notebook we finally point an LSTM at a *real* series: the classic monthly airline-passengers dataset (144 months, 1949–1960). It is tiny — which is exactly why it is a great teacher. Everything that can go right or wrong with a recurrent forecaster (order, leakage, baselines, error compounding) shows up here in a few hundred training epochs.

**Learning goals**

- Frame forecasting as a **many-to-one regression** problem: predict $\hat{x}_t$ from the window $(x_{t-L}, \dots, x_{t-1})$.
- Explain why **order** is the entire signal in a time series — and why shuffling one is fatal.
- Spot and prevent the two classic **data-leakage** bugs: shuffling the split and fitting scalers on the full series.
- Build sliding windows, train an `nn.LSTM` regressor in PyTorch, and judge it against honest **baselines** (persistence and seasonal-naive).
- Roll the model forward **recursively** for a multi-step forecast and watch errors compound with horizon.

## 1. Framing: forecasting is many-to-one regression

A time series is a sequence $(x_1, x_2, \dots, x_T)$ observed in a fixed order. We turn it into a supervised dataset by sliding a window of length $L$ (the **lookback**) over it:

$$
\hat{x}_t = f_\theta\big(x_{t-L},\; x_{t-L+1},\; \dots,\; x_{t-1}\big)
$$

Each window is one training example; the target is the very next value. One model, many windows — a many-to-one regression, exactly the shape an LSTM produces when you read off its **last** hidden state.

### Why ORDER is the whole point

Tabular ML assumes rows are i.i.d. — interchangeable. A time-series window is the opposite: its information lives in the *arrangement* of the numbers, not in the numbers themselves. The windows $[100, 110, 120]$ and $[120, 110, 100]$ have the same mean and variance, yet opposite meanings (growth vs. decline). A model that cannot see order — or a pipeline that destroys it — cannot tell a trend from its mirror image. That is precisely why we reach for RNN/LSTM/GRU: they consume the window **in sequence**, so position and direction are part of the input.

### The data-leakage pitfall

Two rules, both about keeping the future out of the past:

1. **Never shuffle a time series.** A random train/test split puts future months in the training set and past months in the test set. The model then "predicts" months whose neighbours it has already memorized — test error comes out optimistically, dishonestly low. Split **chronologically** instead: train on the past, test on the future.
2. **Never fit scalers on the full series.** Fitting min/max (or mean/std) on all 144 months lets *future* values determine the transformation of the *past* — the network sees a history that was normalized using knowledge of the future. Fit the scaler on the **training slice only**, then *apply* it to everything.

In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

# ---- Config: the knobs you are invited to turn ----
DATA_URL = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv"
LOOKBACK = 12        # months of history per input window (one full year of seasonality)
HIDDEN = 64          # LSTM hidden size
EPOCHS = 300         # dataset is tiny, so full-batch training for a few hundred epochs is cheap
LR = 1e-2
TRAIN_FRAC = 0.8     # chronological split: first 80% of months for training
FORECAST_STEPS = 24  # recursive multi-step horizon past the training cutoff

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

## 2. Data: the classic airline-passengers series

Monthly totals of international airline passengers, in thousands, January 1949 – December 1960. One small gotcha: some versions of this CSV end with a **non-data footer line** (`International airline passengers: monthly totals ...`). Hard-coding `skipfooter=1` would fix that but *silently delete the last real month* whenever the footer is absent — so instead we parse the `Month` column defensively and keep only the rows that are real dates. That works whether or not the footer is there.

> **What to notice in the plot.** Two structures jump out, and they are the whole problem: a steady **upward trend** (air travel grew through the 1950s) and a strong **yearly seasonality** (summer peaks, winter troughs). Also notice that the seasonal swings *grow* with the level — the series is roughly multiplicative, which makes plain min-max scaling imperfect and the forecasting task honestly hard.

In [ ]:
df = pd.read_csv(DATA_URL)
# Defensive parse: some versions of this file end with a non-data footer line.
# Coerce Month to dates and drop anything that is not a real month.
df["Month"] = pd.to_datetime(df["Month"], errors="coerce")
df = df.dropna(subset=["Month"]).reset_index(drop=True)
passengers = df["Passengers"].to_numpy(dtype=np.float32)

print(df.head(3))
print(df.tail(3))
print(f"{len(df)} monthly observations, "
      f"{df['Month'].iloc[0]:%b %Y} to {df['Month'].iloc[-1]:%b %Y}")

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(df["Month"], passengers, marker="o", markersize=3)
ax.set(title="Monthly international airline passengers (thousands), 1949-1960",
       xlabel="Month", ylabel="Passengers (thousands)")
ax.grid(alpha=0.3)
plt.show()

## 3. Windows, scaling, and the split — engineered to not leak

Three choices, each aimed at a specific failure mode:

1. **`LOOKBACK = 12`.** With monthly data and yearly seasonality, a 12-month window means the model always sees each calendar month exactly once — the minimum context that can express the seasonal pattern.
2. **Scaler fit on the training slice only.** We min-max scale using the min and max of the *training* months, then apply that fixed transform to the whole series. The past is transformed without any knowledge of the future.
3. **Chronological 80/20 split, assigned by target index.** A window belongs to training iff its **target** month lies in the first 80% of the series. Test windows may *look back* into the training period for their inputs — that is legitimate, because at prediction time you genuinely do know last year's values. What must never happen is a *target* from the future leaking into training.

> **What to notice in the numbers below.** Because the trend keeps rising, the scaled *test* values exceed $1.0$ — the model is asked to extrapolate beyond anything it saw in training. Keep that in mind when you judge the fit later: it is the price of an honest, leakage-free split.

In [ ]:
n = len(passengers)
cutoff = int(n * TRAIN_FRAC)  # first index that belongs to the test period

# Rule 2: fit the scaler on the TRAINING slice only, then apply to everything.
train_values = passengers[:cutoff]
data_min, data_max = train_values.min(), train_values.max()

def scale(x):
    return (x - data_min) / (data_max - data_min)

def unscale(x):
    return x * (data_max - data_min) + data_min

scaled = scale(passengers)

# Sliding windows: inputs = scaled[t-L : t], target = scaled[t].
X_all, y_all = [], []
for t in range(LOOKBACK, n):
    X_all.append(scaled[t - LOOKBACK:t])
    y_all.append(scaled[t])

X_all = torch.tensor(np.array(X_all), dtype=torch.float32).unsqueeze(-1)  # (N, LOOKBACK, 1)
y_all = torch.tensor(np.array(y_all), dtype=torch.float32).unsqueeze(-1)  # (N, 1)

# Rule 1: chronological split — a window is "train" iff its TARGET index < cutoff.
target_indices = np.arange(LOOKBACK, n)   # index of the target month of each window
train_mask = target_indices < cutoff
X_train, y_train = X_all[train_mask], y_all[train_mask]
X_test, y_test = X_all[~train_mask], y_all[~train_mask]

print(f"train windows: {len(X_train)}, test windows: {len(X_test)}")
print(f"cutoff month: {df['Month'].iloc[cutoff]:%b %Y} — everything from here on is test")
print(f"scaled test values range: {scaled[cutoff]:.2f} .. {scaled.max():.2f}  (above 1.0 = extrapolation)")

## 4. Model: an LSTM many-to-one regressor

The architecture is deliberately minimal:

$$
(x_{t-12}, \dots, x_{t-1}) \;\xrightarrow{\;\text{LSTM}\;}\; h_{t-1} \;\xrightarrow{\;\text{Linear}\;}\; \hat{x}_t
$$

- `nn.LSTM(input_size=1, hidden_size=64, batch_first=True)` consumes the window one month at a time; its input shape is `(batch, 12, 1)`.
- We read off the **last hidden state** $h_{t-1}$ — a 64-dimensional summary of the whole window — and map it to one number with `nn.Linear(64, 1)`.
- Loss is **MSE** in scaled units, optimizer **Adam**. We train full-batch (all 103 windows at once) for a few hundred epochs; the dataset is tiny, so this takes seconds.

In [ ]:
class LSTMForecaster(nn.Module):
    # Many-to-one LSTM regressor: a window of LOOKBACK values -> one prediction.

    def __init__(self, hidden_size: int):
        super().__init__()
        self.lstm = nn.LSTM(input_size=1, hidden_size=hidden_size, batch_first=True)
        self.head = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # x: (batch, LOOKBACK, 1)
        output, (h_n, c_n) = self.lstm(x)
        last_hidden = h_n[-1]           # (batch, hidden) — summary of the whole window
        return self.head(last_hidden)   # (batch, 1)


model = LSTMForecaster(HIDDEN)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
loss_fn = nn.MSELoss()

losses = []
for epoch in range(1, EPOCHS + 1):
    model.train()
    optimizer.zero_grad()
    loss = loss_fn(model(X_train), y_train)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
    if epoch == 1 or epoch % 50 == 0:
        print(f"epoch {epoch:>4d}/{EPOCHS}  train MSE (scaled) = {loss.item():.6f}")

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(losses)
ax.set(title="Training loss", xlabel="Epoch", ylabel="MSE (scaled units)", yscale="log")
ax.grid(alpha=0.3)
plt.show()

## 5. Evaluation — and the humility of baselines

We report **RMSE in thousands of passengers** (original units, so the number means something). A neural network's score is meaningless until it is compared against **baselines that require no learning at all**:

- **Persistence:** $\hat{x}_t = x_{t-1}$ — "next month equals this month."
- **Seasonal-naive:** $\hat{x}_t = x_{t-12}$ — "next month equals the same month last year." On strongly seasonal data this is a *genuinely strong* baseline: it gets the seasonality exactly right and only misses the trend.

> **What to notice.** Do not be surprised if seasonal-naive comes close to the LSTM — on this dataset it often does. That is not a failure of the notebook; it **is** the lesson. With 103 training windows, a single scalar input, and a rising trend that pushes the test period outside the scaled range, a deep model has little room to show off. The correct first question of any forecasting project is never "which neural architecture?" but "does my model actually beat `x[t-12]`?"

In [ ]:
model.eval()
with torch.no_grad():
    pred_train = unscale(model(X_train).squeeze(1).numpy())
    pred_test = unscale(model(X_test).squeeze(1).numpy())

train_target_idx = target_indices[train_mask]
test_target_idx = target_indices[~train_mask]

def rmse(a, b):
    return float(np.sqrt(np.mean((np.asarray(a) - np.asarray(b)) ** 2)))

actual_test = passengers[test_target_idx]

# Baseline 1 — persistence: predict the last observed value.
pred_persistence = passengers[test_target_idx - 1]
# Baseline 2 — seasonal-naive: predict the value 12 months earlier.
pred_seasonal = passengers[test_target_idx - LOOKBACK]

print(f"LSTM            test RMSE: {rmse(pred_test, actual_test):6.1f} thousand passengers")
print(f"persistence     test RMSE: {rmse(pred_persistence, actual_test):6.1f} thousand passengers")
print(f"seasonal-naive  test RMSE: {rmse(pred_seasonal, actual_test):6.1f} thousand passengers")

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(df["Month"], passengers, label="Actual", color="black", lw=1.5)
ax.plot(df["Month"].iloc[train_target_idx], pred_train, label="LSTM fit (train)", alpha=0.8)
ax.plot(df["Month"].iloc[test_target_idx], pred_test, label="LSTM one-step forecast (test)", lw=2)
ax.plot(df["Month"].iloc[test_target_idx], pred_seasonal, label="seasonal-naive (test)", ls="--", alpha=0.7)
ax.axvline(df["Month"].iloc[cutoff], color="red", ls=":", label="train/test cutoff")
ax.set(title="One-step-ahead fits on train and test", ylabel="Passengers (thousands)")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

## 6. Recursive multi-step forecasting: feeding the model its own output

The evaluation above was **one-step-ahead**: every test prediction still received a window of *true* history. A real forecast must go further. To predict 24 months past the training cutoff we:

1. seed the model with the last 12 months of *training* history,
2. predict the next month, append **that prediction** to the window, drop the oldest month,
3. repeat — the model now consumes its own outputs.

$$
\hat{x}_{t} = f_\theta(\hat{x}_{t-12}, \dots, \hat{x}_{t-1}) \quad \text{(estimates feeding estimates)}
$$

This is where errors **compound**: a mistake at horizon $k$ becomes an *input* at horizon $k{+}1$, so small biases can snowball — the seasonal phase can drift, and the level can sag or run away. Because the airline series is seasonal, the error need not grow monotonically (forecasts that land back in phase can be locally better), but the overall trend is clear: **the farther out you predict, the less you should trust it.**

In [ ]:
model.eval()
window = scaled[cutoff - LOOKBACK:cutoff].copy()  # last 12 months of TRAINING history
recursive_scaled = []
with torch.no_grad():
    for step in range(FORECAST_STEPS):
        x = torch.tensor(window, dtype=torch.float32).view(1, LOOKBACK, 1)
        next_value = model(x).item()
        recursive_scaled.append(next_value)
        window = np.append(window[1:], next_value)  # slide: drop oldest, feed back own prediction

recursive = unscale(np.array(recursive_scaled))
horizon_idx = np.arange(cutoff, cutoff + FORECAST_STEPS)
actual_horizon = passengers[horizon_idx]

print(f"recursive {FORECAST_STEPS}-month RMSE: "
      f"{rmse(recursive, actual_horizon):.1f} thousand passengers")
print(f"(compare: one-step-ahead LSTM test RMSE above — the gap is the cost of self-feeding)")

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(df["Month"], passengers, label="Actual", color="black", lw=1.5)
ax.plot(df["Month"].iloc[horizon_idx], recursive, "o-",
        label=f"LSTM recursive {FORECAST_STEPS}-month forecast", lw=2)
ax.axvline(df["Month"].iloc[cutoff], color="red", ls=":", label="forecast origin (train cutoff)")
ax.set(title="Recursive multi-step forecasting: the model feeds on its own predictions",
       ylabel="Passengers (thousands)")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(np.arange(1, FORECAST_STEPS + 1), np.abs(recursive - actual_horizon), "o-")
ax.set(title="Absolute error vs forecast horizon",
       xlabel="Months ahead", ylabel="|error| (thousands)")
ax.grid(alpha=0.3)
plt.show()

## Try it yourself

1. **GRU instead of LSTM.** Swap `nn.LSTM` for `nn.GRU` (same interface — `forward` returns `(output, h_n)` instead of `(output, (h_n, c_n))`, so adjust one line). Does the simpler cell match the LSTM's test RMSE on this tiny dataset? Which trains faster per epoch, and why?
2. **Longer memory.** Set `LOOKBACK = 24`. The model now sees two full years. Does the test RMSE improve, or do the extra parameters just overfit 91 training windows? Look at both the train fit and the test fit before deciding.
3. **Add a calendar feature.** Give the model a month-of-year one-hot: at each time step, feed `[scaled_value, one_hot(month)]` so `input_size = 13`. The calendar is known for the future, so this is leakage-free. Does explicitly encoding the season help the recursive forecast hold its phase?
4. **Walk-forward validation.** A single 80/20 cutoff judges the model on one particular future. Instead, evaluate at several cutoffs (e.g. 60%, 70%, 80%): retrain from scratch at each, score one-step-ahead RMSE on the following 12 months, and average. How much does the verdict on "LSTM vs seasonal-naive" move around?

## Recap

- Forecasting = **many-to-one regression** over sliding windows: $\hat{x}_t = f_\theta(x_{t-L}, \dots, x_{t-1})$.
- **Order is the signal.** Never shuffle a time series; split chronologically and assign windows by their target index.
- **Leakage hides in preprocessing too.** Fit scalers on the training slice only, then apply.
- A small `nn.LSTM(1, 64)` + linear head trains in seconds and fits the series — but **baselines keep you honest**: seasonal-naive ($\hat{x}_t = x_{t-12}$) is a serious competitor on seasonal data, and beating it is the real bar.
- **Recursive multi-step forecasting** feeds predictions back as inputs; errors compound with horizon, so trust decays the farther out you go.